In [ ]:
import torch
from pathlib import Path

from eval import inference, top_k_accuracy
from model import LeagueDraftModel
from vocabulary import Vocabulary
from data import load_matches, ChampionDataset
from torch.utils.data import DataLoader

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

BATCH_SIZE = 2048

In [3]:
start = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (start, *start.parents)
    if (path / '.git').exists()
)

CHECKPOINT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'checkpoints'
DATA_DIRECTORY = PROJECT_ROOT / 'data' 

In [4]:
checkpoint = torch.load(CHECKPOINT_DIRECTORY / 'best_model.pth', device, weights_only=True)
state_dict = checkpoint['model_state_dict']
champ_dict = checkpoint['riotid_to_name']

vocab = Vocabulary(champ_dict)
model = LeagueDraftModel(len(vocab), vocab.mask_id)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

encoded_matches = load_matches(DATA_DIRECTORY / 'league_data.db', vocab)

test_data = ChampionDataset(encoded_matches, vocab.mask_id)

In [5]:
eval_loader = DataLoader(
    test_data, 
    batch_size = BATCH_SIZE , 
    shuffle=False,
    pin_memory=(device.type=='cuda')
)

In [14]:
print(top_k_accuracy(eval_loader, model, device, 5), top_k_accuracy(eval_loader, model, device, 10))

0.4075501752150366 0.5829882128066263


In [9]:
game = ['malphite', 'diana', 'ahri', 'masked', 'lulu', 'darius', 'warwick', 'orianna', 'ezreal', 'karma']

print(inference(model, game, vocab, device, k=5))

['Yunara', 'Jinx', 'Aphelios', 'Zeri', 'Tristana']
